# Week 04 – CartFlow Raw-Preserving Bronze Ingestion

**Project:** P06 CartFlow — Retail & E-commerce Analytics

**Goal:** Move the five approved batch source files into persistent Bronze Delta tables while preserving source business values and adding technical lineage metadata.

**Week 4 boundary:** Source → Bronze only. No cleansing, deduplication, Silver, DQ/quarantine, Gold, Power BI, Auto Loader, or streaming.


## Week 4 objective

The completed flow is:

**Approved source files → source inspection → Bronze-ready data → persistent Bronze Delta tables → reconciliation → safe rerun proof**

The project requires one Bronze table per approved batch source, source business values preserved, source filename/ingestion timestamp/run metadata added, source-to-Bronze counts reconciled, and repeat-run behavior proved.


In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField,
    StringType, LongType, DoubleType
)

# This is the same Volume path used by the current CartFlow Week-3 notebook.
# If Catalog Explorer shows a different approved CartFlow Volume, change ONLY this line.
VOLUME_PATH = "/Volumes/cartflow-06/default/cartflow-p06"

RUN_ID = "CARTFLOW-W04-" + spark.sql(
    "SELECT date_format(current_timestamp(), 'yyyyMMdd-HHmmss')"
).first()[0]

SCHEMA_VERSION = "cartflow-bronze-v1"

print("VOLUME_PATH =", VOLUME_PATH)
print("RUN_ID =", RUN_ID)
print("SCHEMA_VERSION =", SCHEMA_VERSION)

display(dbutils.fs.ls(VOLUME_PATH))


## 1. Approved source inventory

In [ ]:
source_inventory = [
    (
        "orders.csv",
        "CSV",
        "orders_source",
        "workspace.default.bronze_ecommerce_orders",
        "order_id",
        "One order lifecycle header"
    ),
    (
        "order_items.parquet",
        "Parquet",
        "order_items_source",
        "workspace.default.bronze_ecommerce_order_items",
        "order_item_id",
        "One product-seller item line"
    ),
    (
        "payments.csv",
        "CSV",
        "payments_source",
        "workspace.default.bronze_ecommerce_payments",
        "payment_id",
        "One payment/installment row"
    ),
    (
        "reviews.csv",
        "CSV",
        "reviews_source",
        "workspace.default.bronze_ecommerce_reviews",
        "review_id",
        "One synthetic eligible review"
    ),
    (
        "sellers.json",
        "JSON Lines",
        "sellers_source",
        "workspace.default.bronze_ecommerce_sellers",
        "seller_id",
        "One fictional seller master row"
    ),
]

display(
    spark.createDataFrame(
        source_inventory,
        [
            "source_file",
            "format",
            "temporary_view",
            "bronze_table",
            "primary_key",
            "approved_grain"
        ]
    )
)


## 2. Verify that all five approved batch files exist

In [ ]:
required_files = [row[0] for row in source_inventory]
volume_files = [item.name.rstrip("/") for item in dbutils.fs.ls(VOLUME_PATH)]

file_check = [(name, name in volume_files) for name in required_files]

display(
    spark.createDataFrame(
        file_check,
        ["required_file", "available"]
    )
)

missing = [name for name, available in file_check if not available]

if missing:
    raise FileNotFoundError(
        "Missing approved CartFlow batch file(s): " + ", ".join(missing)
    )

print("All five approved batch files are available.")


## 3. Explicit schemas for CSV/JSON sources

Bronze keeps source business values as supplied. Casting/standardization and business-rule correction belong to later Silver work.

`order_items.parquet` is read directly with Spark's Parquet reader because the source is already Parquet.


In [ ]:
orders_schema = StructType([
    StructField("source_record_id", StringType(), True),
    StructField("order_id", StringType(), True),
    StructField("customer_region", StringType(), True),
    StructField("customer_state", StringType(), True),
    StructField("customer_city_code", StringType(), True),
    StructField("customer_segment", StringType(), True),
    StructField("order_status", StringType(), True),
    StructField("purchase_ts", StringType(), True),
    StructField("approval_ts", StringType(), True),
    StructField("carrier_handoff_ts", StringType(), True),
    StructField("delivered_ts", StringType(), True),
    StructField("estimated_delivery_ts", StringType(), True),
    StructField("return_ts", StringType(), True),
    StructField("currency_code", StringType(), True),
    StructField("_corrupt_record", StringType(), True),
])

payments_schema = StructType([
    StructField("source_record_id", StringType(), True),
    StructField("payment_id", StringType(), True),
    StructField("order_id", StringType(), True),
    StructField("installment_no", StringType(), True),
    StructField("payment_method", StringType(), True),
    StructField("payment_value", DoubleType(), True),
    StructField("payment_ts", StringType(), True),
    StructField("currency_code", StringType(), True),
    StructField("_corrupt_record", StringType(), True),
])

reviews_schema = StructType([
    StructField("source_record_id", StringType(), True),
    StructField("review_id", StringType(), True),
    StructField("order_id", StringType(), True),
    StructField("review_score", StringType(), True),
    StructField("review_date", StringType(), True),
    StructField("review_sentiment", StringType(), True),
    StructField("_corrupt_record", StringType(), True),
])

sellers_schema = StructType([
    StructField("source_record_id", StringType(), True),
    StructField("seller_id", StringType(), True),
    StructField("seller_region", StringType(), True),
    StructField("seller_state", StringType(), True),
    StructField("seller_type", StringType(), True),
    StructField("service_band", StringType(), True),
    StructField("active_from", StringType(), True),
    StructField("active_to", StringType(), True),
    StructField("seller_status", StringType(), True),
    StructField("_corrupt_record", StringType(), True),
])


## 4. Source 1 — `orders.csv` → `bronze_ecommerce_orders`

In [ ]:
orders_source_df = (
    spark.read
    .format("csv")
    .option("header", "true")
    .option("mode", "PERMISSIVE")
    .option("columnNameOfCorruptRecord", "_corrupt_record")
    .schema(orders_schema)
    .load(f"{VOLUME_PATH}/orders.csv")
)

orders_source_df.createOrReplaceTempView("orders_source")

display(orders_source_df.limit(10))
orders_source_df.printSchema()
print("Source rows:", orders_source_df.count())


In [ ]:
orders_bronze_df = (
    orders_source_df
    .withColumn("_source_file_name", F.lit("orders.csv"))
    .withColumn("_source_file_path", F.lit(f"{VOLUME_PATH}/orders.csv"))
    .withColumn("_ingested_at", F.current_timestamp())
    .withColumn("_ingestion_run_id", F.lit(RUN_ID))
    .withColumn("_schema_version", F.lit(SCHEMA_VERSION))
    .withColumn(
        "_record_hash",
        F.sha2(
            F.concat_ws(
                "||",
                *[
                    F.coalesce(F.col(c).cast("string"), F.lit(""))
                    for c in orders_schema.fieldNames()
                    if c != "_corrupt_record"
                ]
            ),
            256
        )
    )
)

display(orders_bronze_df.limit(10))

(
    orders_bronze_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("workspace.default.bronze_ecommerce_orders")
)

print("Created workspace.default.bronze_ecommerce_orders")


## 5. Source 2 — `order_items.parquet` → `bronze_ecommerce_order_items`

In [ ]:
# Correct reader for the CartFlow Parquet source: Spark reads Parquet directly.
order_items_source_df = (
    spark.read
    .format("parquet")
    .load(f"{VOLUME_PATH}/order_items.parquet")
)

order_items_source_df.createOrReplaceTempView("order_items_source")

display(order_items_source_df.limit(10))
order_items_source_df.printSchema()
print("Source rows:", order_items_source_df.count())


In [ ]:
order_items_bronze_df = (
    order_items_source_df
    .withColumn("_source_file_name", F.lit("order_items.parquet"))
    .withColumn("_source_file_path", F.lit(f"{VOLUME_PATH}/order_items.parquet"))
    .withColumn("_ingested_at", F.current_timestamp())
    .withColumn("_ingestion_run_id", F.lit(RUN_ID))
    .withColumn("_schema_version", F.lit(SCHEMA_VERSION))
    .withColumn(
        "_record_hash",
        F.sha2(
            F.concat_ws(
                "||",
                *[
                    F.coalesce(F.col(c).cast("string"), F.lit(""))
                    for c in order_items_source_df.columns
                ]
            ),
            256
        )
    )
)

display(order_items_bronze_df.limit(10))

(
    order_items_bronze_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("workspace.default.bronze_ecommerce_order_items")
)

print("Created workspace.default.bronze_ecommerce_order_items")


## 6. Source 3 — `payments.csv` → `bronze_ecommerce_payments`

In [ ]:
payments_source_df = (
    spark.read
    .format("csv")
    .option("header", "true")
    .option("mode", "PERMISSIVE")
    .option("columnNameOfCorruptRecord", "_corrupt_record")
    .schema(payments_schema)
    .load(f"{VOLUME_PATH}/payments.csv")
)

payments_source_df.createOrReplaceTempView("payments_source")

display(payments_source_df.limit(10))
payments_source_df.printSchema()
print("Source rows:", payments_source_df.count())


In [ ]:
payments_bronze_df = (
    payments_source_df
    .withColumn("_source_file_name", F.lit("payments.csv"))
    .withColumn("_source_file_path", F.lit(f"{VOLUME_PATH}/payments.csv"))
    .withColumn("_ingested_at", F.current_timestamp())
    .withColumn("_ingestion_run_id", F.lit(RUN_ID))
    .withColumn("_schema_version", F.lit(SCHEMA_VERSION))
    .withColumn(
        "_record_hash",
        F.sha2(
            F.concat_ws(
                "||",
                *[
                    F.coalesce(F.col(c).cast("string"), F.lit(""))
                    for c in payments_schema.fieldNames()
                    if c != "_corrupt_record"
                ]
            ),
            256
        )
    )
)

display(payments_bronze_df.limit(10))

(
    payments_bronze_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("workspace.default.bronze_ecommerce_payments")
)

print("Created workspace.default.bronze_ecommerce_payments")


## 7. Source 4 — `reviews.csv` → `bronze_ecommerce_reviews`

In [ ]:
reviews_source_df = (
    spark.read
    .format("csv")
    .option("header", "true")
    .option("mode", "PERMISSIVE")
    .option("columnNameOfCorruptRecord", "_corrupt_record")
    .schema(reviews_schema)
    .load(f"{VOLUME_PATH}/reviews.csv")
)

reviews_source_df.createOrReplaceTempView("reviews_source")

display(reviews_source_df.limit(10))
reviews_source_df.printSchema()
print("Source rows:", reviews_source_df.count())


In [ ]:
reviews_bronze_df = (
    reviews_source_df
    .withColumn("_source_file_name", F.lit("reviews.csv"))
    .withColumn("_source_file_path", F.lit(f"{VOLUME_PATH}/reviews.csv"))
    .withColumn("_ingested_at", F.current_timestamp())
    .withColumn("_ingestion_run_id", F.lit(RUN_ID))
    .withColumn("_schema_version", F.lit(SCHEMA_VERSION))
    .withColumn(
        "_record_hash",
        F.sha2(
            F.concat_ws(
                "||",
                *[
                    F.coalesce(F.col(c).cast("string"), F.lit(""))
                    for c in reviews_schema.fieldNames()
                    if c != "_corrupt_record"
                ]
            ),
            256
        )
    )
)

display(reviews_bronze_df.limit(10))

(
    reviews_bronze_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("workspace.default.bronze_ecommerce_reviews")
)

print("Created workspace.default.bronze_ecommerce_reviews")


## 8. Source 5 — `sellers.json` → `bronze_ecommerce_sellers`

In [ ]:
# sellers.json is JSON Lines: one JSON object per physical line.
sellers_source_df = (
    spark.read
    .format("json")
    .option("multiLine", "false")
    .schema(sellers_schema)
    .load(f"{VOLUME_PATH}/sellers.json")
)

sellers_source_df.createOrReplaceTempView("sellers_source")

display(sellers_source_df.limit(10))
sellers_source_df.printSchema()
print("Source rows:", sellers_source_df.count())


In [ ]:
sellers_bronze_df = (
    sellers_source_df
    .withColumn("_source_file_name", F.lit("sellers.json"))
    .withColumn("_source_file_path", F.lit(f"{VOLUME_PATH}/sellers.json"))
    .withColumn("_ingested_at", F.current_timestamp())
    .withColumn("_ingestion_run_id", F.lit(RUN_ID))
    .withColumn("_schema_version", F.lit(SCHEMA_VERSION))
    .withColumn(
        "_record_hash",
        F.sha2(
            F.concat_ws(
                "||",
                *[
                    F.coalesce(F.col(c).cast("string"), F.lit(""))
                    for c in sellers_schema.fieldNames()
                    if c != "_corrupt_record"
                ]
            ),
            256
        )
    )
)

display(sellers_bronze_df.limit(10))

(
    sellers_bronze_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("workspace.default.bronze_ecommerce_sellers")
)

print("Created workspace.default.bronze_ecommerce_sellers")


## 9. Verify all Bronze tables and technical metadata

In [ ]:
bronze_tables = [
    "workspace.default.bronze_ecommerce_orders",
    "workspace.default.bronze_ecommerce_order_items",
    "workspace.default.bronze_ecommerce_payments",
    "workspace.default.bronze_ecommerce_reviews",
    "workspace.default.bronze_ecommerce_sellers",
]

for table_name in bronze_tables:
    print("\n---", table_name, "---")
    df = spark.table(table_name)
    print("Row count:", df.count())
    print("Columns:", df.columns)
    display(df.limit(5))


## 10. Consolidated source-to-Bronze reconciliation

In [ ]:
reconciliation = [
    (
        "orders.csv",
        orders_source_df.count(),
        spark.table("workspace.default.bronze_ecommerce_orders").count()
    ),
    (
        "order_items.parquet",
        order_items_source_df.count(),
        spark.table("workspace.default.bronze_ecommerce_order_items").count()
    ),
    (
        "payments.csv",
        payments_source_df.count(),
        spark.table("workspace.default.bronze_ecommerce_payments").count()
    ),
    (
        "reviews.csv",
        reviews_source_df.count(),
        spark.table("workspace.default.bronze_ecommerce_reviews").count()
    ),
    (
        "sellers.json",
        sellers_source_df.count(),
        spark.table("workspace.default.bronze_ecommerce_sellers").count()
    ),
]

recon_df = (
    spark.createDataFrame(
        reconciliation,
        ["source_file", "source_count", "bronze_count"]
    )
    .withColumn(
        "count_difference",
        F.col("bronze_count") - F.col("source_count")
    )
    .withColumn(
        "status",
        F.when(F.col("count_difference") == 0, "MATCH")
         .otherwise("CHECK")
    )
)

display(recon_df)

if recon_df.filter(F.col("status") != "MATCH").count() > 0:
    raise RuntimeError("At least one source/Bronze count does not match.")


## 11. Check required metadata columns

In [ ]:
required_metadata = [
    "_source_file_name",
    "_source_file_path",
    "_ingested_at",
    "_ingestion_run_id",
    "_schema_version",
    "_record_hash",
]

metadata_results = []

for table_name in bronze_tables:
    columns = spark.table(table_name).columns
    missing = [c for c in required_metadata if c not in columns]
    metadata_results.append((table_name, ", ".join(missing) if missing else "NONE"))

metadata_df = spark.createDataFrame(
    metadata_results,
    ["bronze_table", "missing_required_metadata"]
)

display(metadata_df)

if metadata_df.filter(F.col("missing_required_metadata") != "NONE").count() > 0:
    raise RuntimeError("One or more Bronze tables are missing required metadata.")


## 12. Demonstrate source-value preservation

In [ ]:
display(
    spark.table("workspace.default.bronze_ecommerce_orders")
    .select(
        "source_record_id",
        "order_id",
        "customer_region",
        "customer_state",
        "order_status",
        "purchase_ts",
        "_source_file_name",
        "_ingested_at",
        "_ingestion_run_id",
        "_schema_version",
        "_record_hash"
    )
    .limit(10)
)


## 13. Safe rerun / idempotence proof

The Bronze tables are written with `mode("overwrite")`.

To prove repeat-run behavior:

1. Re-run the complete **Orders source read** cell.
2. Re-run the complete **Orders Bronze write** cell.
3. Then execute the proof cell below.
4. The Bronze count should still equal the source count rather than doubling.


In [ ]:
orders_source_count_after_rerun = orders_source_df.count()
orders_bronze_count_after_rerun = (
    spark.table("workspace.default.bronze_ecommerce_orders").count()
)

rerun_result = (
    spark.createDataFrame(
        [
            (
                "orders.csv",
                orders_source_count_after_rerun,
                orders_bronze_count_after_rerun
            )
        ],
        [
            "source_file",
            "source_count_after_rerun",
            "bronze_count_after_rerun"
        ]
    )
    .withColumn(
        "status",
        F.when(
            F.col("source_count_after_rerun") ==
            F.col("bronze_count_after_rerun"),
            "SAFE_RERUN_MATCH"
        ).otherwise("CHECK")
    )
)

display(rerun_result)


## 14. Delta history for the rerun-tested Bronze table

In [ ]:
spark.sql(
    "DESCRIBE HISTORY workspace.default.bronze_ecommerce_orders"
).select(
    "version",
    "timestamp",
    "operation",
    "operationParameters"
).show(truncate=False)


## Week 4 exit checklist

- [ ] All five approved batch source files are covered.
- [ ] Correct reader is used for every format.
- [ ] Parquet is read directly with Spark.
- [ ] Source business values are preserved.
- [ ] One persistent Bronze Delta table exists for every approved batch source.
- [ ] Required technical metadata is present.
- [ ] Source and Bronze counts reconcile.
- [ ] Orders rerun is proved not to double the Bronze row count.
- [ ] Delta history is shown.
- [ ] Only genuine execution evidence is saved.
- [ ] Week 4 log and AI Transparency Note are updated.
- [ ] No Silver, DQ/quarantine, Gold, Power BI, Auto Loader or streaming work is included.

**Important:** Do not type expected results such as `100000` or `MATCH` into the notebook as if they were execution evidence. Run the cells and keep the real Databricks outputs.
